# AI policy pipeline: one test

This notebook runs **one test**: which syllabi, which model, which cutoffs (block **1 · Settings**). It only **runs** the steps and shows their results; each step's code is in `[02] Pipeline/functions/`, and the shared settings are in `settings.py` there.

```
[01] Data ─▶ 1 Input ─▶ 2 Screen ─▶ 3 Chunk ─┬▶ 4 Code ────┬▶ 5 Rules ─▶ 6 Export ─▶ Test A–C
             Python     Kev (GPU)   Python    └▶ 4b Regions ┘  Python     pandas      labels.csv
                                               Kev (GPU)      (4b is compared in Test B; step 5 does not read it yet)
```

**How to run it:** `[01] Data`, `[02] Pipeline`, `[03] Fine tuning` and `[04] Test` sit in one project folder. Open this notebook from its test folder on a machine with an NVIDIA GPU, and run every block.

**What each step block shows:** a header (what it reads and writes) · a live progress bar · one result line (✓ done, ⏸ waiting and why, ✗ stopped) · its results as a table. **2 · Setup** and the end of the notebook show a **status board** of every step.

Everything is saved in this test's `Output` as it goes: a dropped session loses nothing, a finished step is skipped next time, and a step runs again by itself when a setting, a function it uses, or the model changes. Without a GPU the two Kev steps wait and everything else still runs.

## 1 · Settings
The only block you edit. Everything shared by all tests is in `[02] Pipeline/functions/settings.py`.

In [ ]:
# ---------- this test: the only lines that differ between tests ----------
TEST = "<test name>"                 # this test's folder inside [04] Test, e.g. "UNI labelled Kev-4B"; results go in its Output
UNIVERSITIES = ["UNI"]                # which universities, by short code (UNIVERSITY_NAMES in settings.py gives their folders)
LABELLED_ONLY = True                  # True: only the hand-coded syllabi (<University>/labelled/); False: every syllabus
MODEL = "jaredpalmer/kev-4b"          # Kev's model: a Hugging Face name, or the folder of a fine-tuned checkpoint
CUTOFFS = {                           # yours to set for this model, once a labelled run shows what each cutoff gives
    "policy_found":      None,
    "cs_present":        None,
    "bp_present":        None,
    "prohibited":        None,
    "specific_context":  None,
    "ask_instructor":    None,
    "citation_required": None,
}
TRY_FIRST = 3                         # Kev steps: try this many shortest and longest syllabi before the full run (0 = skip)

## 2 · Setup
Connects Drive, loads the pipeline's functions, starts Kev when there is a GPU, and shows where this test stands.

In [ ]:
import importlib
import sys
from pathlib import Path

sys.dont_write_bytecode = True               # keeps Python from adding a __pycache__ folder to [02] Pipeline
DRIVE_PROJECT = "/content/drive/MyDrive/AI Syllabus Project"     # the project folder in My Drive

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_PROJECT)
else:                                        # on the Mac: the folder above this notebook that holds [02] Pipeline
    ROOT = Path.cwd()
    while not (ROOT / "[02] Pipeline").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if not (ROOT / "[02] Pipeline" / "functions").exists():
    raise FileNotFoundError("No '[02] Pipeline/functions' in " + str(ROOT) + ". In Colab, put [01] Data, [02] Pipeline, [03] Fine tuning "
                            "and [04] Test in My Drive/AI Syllabus Project first.")
sys.path.insert(0, str(ROOT / "[02] Pipeline" / "functions"))

import settings, helpers, show, server_kev, labels, metrics
import s1_input, s2_screen_kev, s3_chunk, s4_code_kev, s4b_regions_kev, s5_rules, s6_export

# Running this block again picks up .py files that changed since. server_kev is left as it is: it holds the running Kev.
for module in [settings, helpers, show, labels, metrics, s1_input, s2_screen_kev, s3_chunk, s4_code_kev, s4b_regions_kev,
               s5_rules, s6_export]:
    importlib.reload(module)

TEST_FOLDER = settings.TESTS / TEST
if not TEST_FOLDER.exists():
    raise FileNotFoundError("No test folder " + str(TEST_FOLDER) + ". TEST must name this notebook's own folder in [04] Test.")
helpers.start(TEST_FOLDER / "Output", settings.DATA)
server_kev.start(MODEL)                      # needs a GPU; without one this only says so

print("Test:", TEST, "|", ", ".join(UNIVERSITIES), "|", "hand-coded syllabi only" if LABELLED_ONLY else "every syllabus",
      "|", MODEL, "| on", "Colab" if helpers.ON_COLAB else "this computer")
print("Results go in:", helpers.OUTPUT)
show.status_board(server_kev.is_up(), None not in CUTOFFS.values(), UNIVERSITIES)

## Step 1 · Input
Each university's syllabi become one jsonl in `[01] Data`: `<University>/syllabi.jsonl`, or `<University>/labelled/labelled.jsonl` for the hand-coded set. One row per syllabus, with emails and phone numbers replaced. It is rebuilt only when its text files or step 1's code changed; otherwise this block only checks it, and the later steps read it directly (no copy in `Output`). For a full set (`LABELLED_ONLY = False`), any PDF or Word file with no text file yet is converted into `txt/` first.

In [ ]:
show.header(1)
if not LABELLED_ONLY:
    for university in UNIVERSITIES:
        s1_input.convert_missing(university)
        s1_input.write_syllabi_csv(university)
report = helpers.build_input(s1_input, {"UNIVERSITIES": UNIVERSITIES, "LABELLED_ONLY": LABELLED_ONLY})
show.report(report)
show.input_summary(helpers.read("1 input"), UNIVERSITIES, report)

## Step 2 · Screen (Kev)
Kev reads the whole syllabus and answers one question: is there an AI policy? The answer is a probability between 0 and 1.

In [ ]:
show.header(2, MODEL)
if TRY_FIRST and server_kev.is_up() and not helpers.ready(["2 screen"]):
    show.tried(helpers.try_first(s2_screen_kev.screen_syllabus, helpers.shortest_and_longest(helpers.read("1 input"), TRY_FIRST)))
report = helpers.run_step("2 screen", s2_screen_kev.screen_syllabus, reads="1 input", server=server_kev, model=MODEL,
                          progress=show.bar)
show.report(report)
show.probabilities("2 screen")

## Step 3 · Chunk
Finds the policy text with plain Python, no model: every sentence with a keyword, plus `WINDOW` sentences on each side. Regions that touch are merged; separate ones are joined with `[...]`.

In [ ]:
show.header(3)
report = helpers.run_step("3 chunks", s3_chunk.chunk_syllabus, reads="1 input", progress=show.bar)
show.report(report)
show.chunks_summary(helpers.read("3 chunks"))
# to read one syllabus's chunk:   show.chunk("UNI-0001")

## Step 4 · Code (Kev)
Kev reads only the chunk and answers the 6 questions in one call. Syllabi with no chunk are skipped.

In [ ]:
show.header(4, MODEL)
if TRY_FIRST and server_kev.is_up() and not helpers.ready(["4 code"]):
    to_code = [row for row in helpers.read("3 chunks") if s4_code_kev.needs_coding(row)]
    show.tried(helpers.try_first(s4_code_kev.code_syllabus, helpers.shortest_and_longest(to_code, TRY_FIRST, "chunk_chars"),
                                 "chunk_chars"))
report = helpers.run_step("4 code", s4_code_kev.code_syllabus, reads="3 chunks", only=s4_code_kev.needs_coding,
                          server=server_kev, model=MODEL, progress=show.bar)
show.report(report)
show.probabilities("4 code")

## Step 4b · Regions (Kev)
Kev reads each region of the chunk on its own (the parts step 3 joined with `[...]`) and answers CS and BP for it, one call per region. Test B compares it with step 4; step 5 does not use it yet.

In [ ]:
show.header("4b", MODEL)
if TRY_FIRST and server_kev.is_up() and not helpers.ready(["4b regions"]):
    to_split = [row for row in helpers.read("3 chunks") if s4b_regions_kev.needs_regions(row)]
    show.tried(helpers.try_first(s4b_regions_kev.code_regions, helpers.shortest_and_longest(to_split, TRY_FIRST, "chunk_chars"),
                                 "chunk_chars"))
report = helpers.run_step("4b regions", s4b_regions_kev.code_regions, reads="3 chunks", only=s4b_regions_kev.needs_regions,
                          server=server_kev, model=MODEL, progress=show.bar)
show.report(report)
show.regions_summary("4b regions")

## Step 5 · Rules
Turns the probabilities into the codebook's answers, 1 / 0 / n/a, with this test's `CUTOFFS` (block 1).

In [ ]:
show.header(5)
waiting = helpers.waiting_for(["2 screen", "3 chunks", "4 code"])
if None in CUTOFFS.values():
    show.waiting(["CUTOFFS (block 1) still has empty values; Test B shows what each cutoff would give"])
elif waiting:
    show.waiting(waiting)
else:
    decisions = [s5_rules.apply_rules(row, CUTOFFS) for row in helpers.join(["1 input", "2 screen", "3 chunks", "4 code"])]
    show.report(helpers.write("5 decisions", decisions, made_from=["2 screen", "3 chunks", "4 code"]))
    show.decisions_summary(decisions)

## Step 6 · Export
One CSV per university in `Output`, one row per syllabus, sorted so it can be split by subject.

In [ ]:
show.header(6)
waiting = helpers.waiting_for(["5 decisions"])
if waiting:
    show.waiting(waiting)
else:
    rows = helpers.join(["1 input", "5 decisions"])
    saved = []
    for university in UNIVERSITIES:
        table = s6_export.export(rows, university)
        helpers.save_output(university, table)
        saved.append((university, len(table)))
    show.exports(saved)

## Test
Compares the pipeline with the hand-coded labels (`[01] Data/<University>/labelled/labels.csv`). Every table it makes is saved in `Output/test_results/`. A university nobody hand-coded has none, and then these blocks only say so. Test A works as soon as step 3 is done; B and C need Kev's answers.

In [ ]:
show.header("A")
gold = labels.read_gold(UNIVERSITIES, settings.GOLD_COLUMNS)
for label in gold:
    label["policy_found"] = labels.gold_policy_found(label)
waiting = helpers.waiting_for(["3 chunks"])
if not gold:
    show.waiting(["hand-coded labels: there are none for " + ", ".join(UNIVERSITIES)])
elif waiting:
    show.waiting(waiting)
else:
    check = metrics.chunk_check(labels.attach(helpers.join(["1 input", "3 chunks"]), gold))
    helpers.save_output("chunk check", check, settings.TEST_RESULTS)
    show.chunk_check(check)

In [ ]:
show.header("B")
waiting = helpers.waiting_for(["2 screen", "3 chunks", "4 code"])
if not gold:
    show.waiting(["hand-coded labels: there are none for " + ", ".join(UNIVERSITIES)])
elif waiting:
    show.waiting(waiting)
else:
    rows = labels.attach(helpers.join(["1 input", "2 screen", "3 chunks", "4 code"]), gold)
    fields = ["policy_found"] + settings.CODE_FIELDS
    by_field, by_university = metrics.score(rows, fields, settings.GOLD_NA_COUNTS_AS_NO)
    cutoffs = metrics.cutoff_table(rows, fields, settings.GOLD_NA_COUNTS_AS_NO)
    helpers.save_output("scores by field", by_field, settings.TEST_RESULTS)
    if len(by_university) > 0:
        helpers.save_output("scores by university", by_university, settings.TEST_RESULTS)
    helpers.save_output("cutoff tables", cutoffs, settings.TEST_RESULTS)
    print("Scores by field:")
    display(by_field)
    print("What each cutoff would give for policy_found (all fields: cutoff tables.csv):")
    display(cutoffs[cutoffs["field"] == "policy_found"])

    # step 4b: CS / BP per region against the whole chunk
    if helpers.ready(["4b regions"]):
        rows = labels.attach(helpers.join(["1 input", "2 screen", "3 chunks", "4 code", "4b regions"]), gold)
        regions = metrics.region_comparison(rows, settings.GOLD_NA_COUNTS_AS_NO)
        helpers.save_output("region comparison", regions, settings.TEST_RESULTS)
        print("CS / BP: whole chunk (step 4) against each region (4b):")
        display(regions)
    else:
        show.waiting(helpers.waiting_for(["4b regions"]) + ["(only for the region comparison)"])

In [ ]:
show.header("C")
waiting = helpers.waiting_for(["2 screen", "4 code"])
if waiting:
    show.waiting(waiting)
else:
    speed = metrics.timing(helpers.join(["1 input", "2 screen", "4 code", "4b regions"]))
    helpers.save_output("timing", speed, settings.TEST_RESULTS)
    display(speed)

## Status
Where this test stands now, step by step.

In [ ]:
show.status_board(server_kev.is_up(), None not in CUTOFFS.values(), UNIVERSITIES)